# Diabetes mellitus prediction

In [11]:
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from diabetes_prediction import (
    DiabetesClassifier,
    DiabetesDataLoader,
    DropMissingRows,
    MeanImputer,
    BinarizeFeature,
    OneHotFeature,
)


In [12]:
DATA_PATH = Path("sample_diabetes_mellitus_data.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../sample_diabetes_mellitus_data.csv")

REQUIRED_COLUMNS = ["age", "gender", "ethnicity"]
MEAN_FILL_COLUMNS = ["height", "weight"]
FEATURE_COLUMNS = [
    "age",
    "height",
    "weight",
    "aids",
    "cirrhosis",
    "hepatic_failure",
    "immunosuppression",
    "leukemia",
    "lymphoma",
    "solid_tumor_with_metastasis",
]
TARGET_COLUMN = "diabetes_mellitus"

## a. Load the data

In [13]:
loader = DiabetesDataLoader(DATA_PATH, test_size=0.2, random_state=42)
patients_df = loader.read()
patients_df.shape


(10000, 52)

## b. Split the data between train and test

In [14]:
train_df, test_df = loader.split(patients_df)
train_df.shape, test_df.shape


((8000, 52), (2000, 52))

## c. Remove rows with NaN in age, gender and ethnicity

In [15]:
missing_row_remover = DropMissingRows(columns=REQUIRED_COLUMNS)
train_df = missing_row_remover.transform(train_df)
test_df = missing_row_remover.transform(test_df)


## d. Fill NaN in height and weight with the mean

In [16]:
mean_imputer = MeanImputer(columns=MEAN_FILL_COLUMNS)
train_df = mean_imputer.fit_transform(train_df)
test_df = mean_imputer.transform(test_df)


## e. One hot encoding of ethnicity

In [17]:
ethnicity_encoder = OneHotFeature("ethnicity")
train_df = ethnicity_encoder.fit_transform(train_df)
test_df = ethnicity_encoder.transform(test_df)


## f. Binary variable for gender (F = 1, M = 0)

In [18]:
gender_encoder = BinarizeFeature("gender", positive_value="F")
train_df = gender_encoder.fit_transform(train_df)
test_df = gender_encoder.transform(test_df)


## g. Train the model

In [19]:
classifier = DiabetesClassifier(
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    model=RandomForestClassifier(max_depth=5, random_state=42),
)
classifier.train(train_df)


## h. Predict on train and test

In [20]:
train_df["predictions"] = classifier.predict(train_df)
test_df["predictions"] = classifier.predict(test_df)
test_df[[TARGET_COLUMN, "predictions"]].head()


,diabetes_mellitus,predictions
6252,1,0.334438
1731,0,0.209174
4742,0,0.143801
4521,0,0.160066
6340,1,0.293970


## i. Train and test ROC AUC

In [21]:
train_roc_auc = roc_auc_score(train_df[TARGET_COLUMN], train_df["predictions"])
test_roc_auc = roc_auc_score(test_df[TARGET_COLUMN], test_df["predictions"])

print(f"Train ROC AUC: {train_roc_auc:.3f}")
print(f"Test ROC AUC: {test_roc_auc:.3f}")


Train ROC AUC: 0.715
Test ROC AUC: 0.653
